## Example 1: COG profiles

In [ ]:
# Example 1: COG profiles
from rio_cogeo import cog_profiles

print("Testing rio-cogeo profiles:")

# Available COG profiles
print(f"✓ Available profiles: {list(cog_profiles.keys())}")

# Show profile details
for name in ['lzw', 'deflate', 'jpeg']:
    if name in cog_profiles:
        profile = cog_profiles[name]
        print(f"✓ {name.upper()}: compress={profile['compress']}, "
              f"tiled={profile['tiled']}, "
              f"blocksize={profile.get('blockxsize', 'N/A')}")

# Profile structure
sample_profile = cog_profiles['lzw']
print(f"✓ LZW profile keys: {list(sample_profile.keys())}")

assert {'lzw', 'deflate', 'jpeg'} <= set(cog_profiles.keys()), list(cog_profiles.keys())
deflate = cog_profiles.get('deflate')
assert (deflate['driver'], deflate['compress'], deflate['tiled']) == ('GTiff', 'DEFLATE', True), deflate
assert (deflate['blockxsize'], deflate['blockysize']) == (512, 512), deflate

## Example 2: COG translation

In [ ]:
# Example 2: COG translation
from rio_cogeo import cog_translate, cog_validate, cog_info, cog_profiles
from rasterio.transform import from_origin
import tempfile
import rasterio
import numpy as np
import os

print("Testing rio-cogeo translate:")

# A plain 1024 x 1024 striped GeoTIFF: larger than one 512 x 512 block, untiled and without overviews
temp_dir = tempfile.mkdtemp()
src_path = os.path.join(temp_dir, 'input.tif')
dst_path = os.path.join(temp_dir, 'output_cog.tif')
data = (np.arange(1024 * 1024) % 251).astype(np.uint8).reshape(1024, 1024)
with rasterio.open(src_path, 'w', driver='GTiff', height=1024, width=1024, count=1, dtype=np.uint8,
                   crs='EPSG:4326', transform=from_origin(-120, 40, 0.001, 0.001)) as dst:
    dst.write(data, 1)

is_valid, errors, warnings = cog_validate(src_path)
print(f"✓ Input is a valid COG: {is_valid} ({errors})")
assert is_valid is False and any('not tiled' in err for err in errors), errors

# Translate with the DEFLATE profile
cog_translate(src_path, dst_path, cog_profiles.get('deflate'), quiet=True)
is_valid, errors, warnings = cog_validate(dst_path)
print(f"✓ Output is a valid COG: {is_valid} (errors: {errors}, warnings: {warnings})")
assert is_valid is True and errors == [], errors

info = cog_info(dst_path)
print(f"✓ cog_info: COG={info.COG}, compression={info.Compression}, IFDs={[(ifd.Width, ifd.Height) for ifd in info.IFD]}")
assert info.COG is True and info.Compression == 'DEFLATE', info

# The pixels and georeferencing are unchanged; internal tiles and an overview level were added
with rasterio.open(dst_path) as cog:
    print(f"✓ Blocks: {cog.block_shapes}, overviews: {cog.overviews(1)}, CRS: {cog.crs}")
    assert cog.block_shapes == [(512, 512)], cog.block_shapes
    assert cog.overviews(1) == [2], cog.overviews(1)
    assert cog.crs.to_epsg() == 4326 and cog.transform == from_origin(-120, 40, 0.001, 0.001)
    assert np.array_equal(cog.read(1), data)

os.unlink(src_path)
os.unlink(dst_path)

## Example 3: COG validation

In [ ]:
# Example 3: COG validation
from rio_cogeo import cog_validate
import tempfile
import rasterio
import numpy as np
import os

print("Testing rio-cogeo validation:")

# Create sample GeoTIFF
temp_file = tempfile.NamedTemporaryFile(suffix='.tif', delete=False)
data = np.random.randint(0, 255, (100, 100), dtype=np.uint8)

with rasterio.open(temp_file.name, 'w', driver='GTiff', height=100, width=100, count=1, dtype=np.uint8, tiled=True, blockxsize=256, blockysize=256) as dst:
    dst.write(data, 1)

print(f"✓ Created test file: {temp_file.name}")
print(f"✓ cog_validate function: {type(cog_validate).__name__}")

# Validate the file
is_valid, errors, warnings = cog_validate(temp_file.name)
print(f"✓ Is valid COG: {is_valid}")
print(f"✓ Errors: {len(errors)}, Warnings: {len(warnings)}")
# A 100 x 100 image fits in one block, so it is a valid COG without overviews
assert is_valid is True and errors == [], errors

# A striped image larger than 512 x 512 is not
big_file = tempfile.NamedTemporaryFile(suffix='.tif', delete=False)
big = np.zeros((1024, 1024), dtype=np.uint8)
with rasterio.open(big_file.name, 'w', driver='GTiff', height=1024, width=1024, count=1, dtype=np.uint8) as dst:
    dst.write(big, 1)
big_valid, big_errors, big_warnings = cog_validate(big_file.name)
print(f"✓ Striped 1024x1024 file valid: {big_valid}, errors: {big_errors}")
assert big_valid is False and big_errors, big_errors

os.unlink(temp_file.name)
os.unlink(big_file.name)

## Example 4: COG info

In [ ]:
# Example 4: COG info
from rio_cogeo import cog_info
import tempfile
import rasterio
import numpy as np

print("Testing rio-cogeo info:")

# Create sample GeoTIFF with overviews
temp_file = tempfile.NamedTemporaryFile(suffix='.tif', delete=False)
data = np.random.randint(0, 255, (512, 512), dtype=np.uint8)

with rasterio.open(temp_file.name, 'w', driver='GTiff', height=512, width=512, count=1, dtype=np.uint8, tiled=True, blockxsize=256, blockysize=256) as dst:
    dst.write(data, 1)

print(f"✓ cog_info function: {type(cog_info).__name__}")

# Get COG info
info = cog_info(temp_file.name)
print(f"✓ Info type: {type(info).__name__}")
print(f"✓ Driver: {info.Driver}")
print(f"✓ Compression: {info.Compression}")
print(f"✓ COG: {info.COG}")

assert info.Driver == 'GTiff' and info.COG is True, info
assert info.Compression is None, info.Compression  # written without compression
profile = info.Profile
print(f"✓ Profile: {profile.Width}x{profile.Height}, {profile.Bands} band(s), {profile.Dtype}, tiled={profile.Tiled}")
assert (profile.Width, profile.Height, profile.Bands, profile.Dtype) == (512, 512, 1, 'uint8'), profile
assert [(ifd.Width, ifd.Height) for ifd in info.IFD] == [(512, 512)], info.IFD

import os
os.unlink(temp_file.name)

## Example 5: Profile customization

In [ ]:
# Example 5: Profile customization
from rio_cogeo import cog_profiles

print("Testing rio-cogeo profile customization:")

# Get base profiles
base_profiles = cog_profiles
print(f"✓ Base profiles: {list(base_profiles.keys())}")

# Create custom profile
custom_profile = base_profiles['lzw'].copy()
custom_profile['blockxsize'] = 512
custom_profile['blockysize'] = 512
custom_profile['compress'] = 'DEFLATE'

print(f"✓ Custom profile created")
print(f"✓ Compression: {custom_profile['compress']}")
print(f"✓ Block size: {custom_profile['blockxsize']}x{custom_profile['blockysize']}")
print(f"✓ Tiled: {custom_profile['tiled']}")

# Compare profiles
print(f"✓ LZW profile compress: {base_profiles['lzw']['compress']}")
print(f"✓ Custom profile compress: {custom_profile['compress']}")

# The copy is independent of the shipped profile
assert custom_profile['compress'] == 'DEFLATE' and custom_profile['tiled'] is True
assert base_profiles['lzw']['compress'] == 'LZW', base_profiles['lzw']